In [ ]:
import pandas as pd
import re

df = pd.read_csv('data_labeled.csv')   # dari Tugas 2

print('Dimensi data:', df.shape)
print('\n5 baris pertama:')
print(df.head())
print('\nDistribusi sentimen:')
print(df['sentimen_auto'].value_counts())

Dimensi data: (4366, 2)

5 baris pertama:
                                       content_clean sentimen_auto
0  aplikasi cukup membantu untuk cek rute dan pos...       positif
1  biasa naik jak dan selalu muncul datanya di ap...       negatif
2  jujur ini ngebantu banget sih buat pejuang tra...       positif
3  aplikasi transjakarta ini jujur ngebantu bange...       positif
4  transjakarta adalah alat yang sangat berguna d...       positif

Distribusi sentimen:
sentimen_auto
positif    3222
netral      669
negatif     475
Name: count, dtype: int64


In [ ]:
!pip install Sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

In [ ]:
import pandas as pd
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Baca kamus
kamus_data = pd.read_csv('kamus_stemming.csv')
kamus_dict = dict(zip(kamus_data['kata_tidak_baku'].astype(str), kamus_data['kata_baku'].astype(str)))

stemmer = StemmerFactory().create_stemmer()
stopword = StopWordRemoverFactory().create_stop_word_remover()

def bersihkan_ke_string_utuh(text):
    if not isinstance(text, str):
        return ""

    # A. Ubah singkatan lewat kamus (Contoh: 'udh' -> 'sudah')
    words = text.split()
    words_baku = [kamus_dict[word] if word in kamus_dict else word for word in words]
    kalimat_baku = " ".join(words_baku)

    # B. Hapus stopword dalam bentuk kalimat utuh (Contoh: 'sudah' langsung DIHAPUS/HILANG)
    kalimat_no_stopword = stopword.remove(kalimat_baku)

    # C. Stemming kalimat utuh (Contoh: 'membantu' -> 'bantu')
    hasil_akhir = stemmer.stem(kalimat_no_stopword)

    return hasil_akhir

# Buat kolom content_processed yang BENAR-BENAR BERSIH
df['content_processed'] = df['content_clean'].apply(bersihkan_ke_string_utuh)

In [ ]:
df.head(10)

,content_clean,sentimen_auto,content_processed
0,aplikasi cukup membantu untuk cek rute dan pos...,positif,aplikasi cukup bantu cek rute posisi bus trans...
1,biasa naik jak dan selalu muncul datanya di ap...,negatif,biasa naik jak selalu muncul data aplikasi sek...
2,jujur ini ngebantu banget sih buat pejuang tra...,positif,jujur ngebantu banget sih buat juang transport...
3,aplikasi transjakarta ini jujur ngebantu bange...,positif,aplikasi transjakarta jujur ngebantu banget si...
4,transjakarta adalah alat yang sangat berguna d...,positif,transjakarta alat sangat guna seringkali perlu...
5,biasa naik jak dan jak dan selalu muncul datan...,negatif,biasa naik jak jak selalu muncul data aplikasi...
6,bintang buat kemudahannya skema rutenya jelas ...,positif,bintang buat mudah skema rute jelas banget nge...
7,membantu bgt utk tracking posisi bis jadi bisa...,positif,bantu bgt utk tracking posisi bis jadi tau est...
8,aplikasinya sangat membantu untuk cek rute dan...,positif,aplikasi sangat bantu cek rute jadwal transjak...
9,ini aplikasi bener ngebantu aku sebagai pendat...,positif,aplikasi bener ngebantu aku datang sih tampil ...


In [ ]:
# 1. Definisikan fungsi tokenize
def tokenize(text):
    # Pastikan data berupa string sebelum di-split
    tokens = str(text).split()
    return tokens

# 2. Terapkan fungsi tokenize ke kolom 'content_processed'
df['tokenize'] = df['content_processed'].apply(tokenize)

# 3. SOLUSI: Filter kolom agar hanya memunculkan ketentuan yang Anda mau
df_hasil = df[['content_clean', 'sentimen_auto', 'content_processed', 'tokenize']]

# 4. Tampilkan 5 data pertama
df_hasil.head()

,content_clean,sentimen_auto,content_processed,tokenize
0,aplikasi cukup membantu untuk cek rute dan pos...,positif,aplikasi cukup bantu cek rute posisi bus trans...,"[aplikasi, cukup, bantu, cek, rute, posisi, bu..."
1,biasa naik jak dan selalu muncul datanya di ap...,negatif,biasa naik jak selalu muncul data aplikasi sek...,"[biasa, naik, jak, selalu, muncul, data, aplik..."
2,jujur ini ngebantu banget sih buat pejuang tra...,positif,jujur ngebantu banget sih buat juang transport...,"[jujur, ngebantu, banget, sih, buat, juang, tr..."
3,aplikasi transjakarta ini jujur ngebantu bange...,positif,aplikasi transjakarta jujur ngebantu banget si...,"[aplikasi, transjakarta, jujur, ngebantu, bang..."
4,transjakarta adalah alat yang sangat berguna d...,positif,transjakarta alat sangat guna seringkali perlu...,"[transjakarta, alat, sangat, guna, seringkali,..."


In [ ]:
import nltk # Import the nltk library
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('indonesian'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# 1. Definisikan fungsi remove_stopwords
def remove_stopwords(text):
    # Memastikan input berupa list, jika berupa string kosong atau nan akan dikembalikan sebagai list kosong
    if not isinstance(text, list):
        return []
    return [word for word in text if word not in stop_words]

# 2. Terapkan fungsi remove_stopwords ke kolom 'tokenize'
# Nama kolom diubah menjadi 'stopword_removal' (menggunakan underscore sesuai standar python)
df['stopword_removal'] = df['tokenize'].apply(remove_stopwords)

# 3. SOLUSI: Menyaring data agar hanya memunculkan 5 kolom yang Anda inginkan secara berurutan
df_hasil = df[['content_clean', 'sentimen_auto', 'content_processed', 'tokenize', 'stopword_removal']]

# 4. Tampilkan 5 data pertama
df_hasil.head()

,content_clean,sentimen_auto,content_processed,tokenize,stopword_removal
0,aplikasi cukup membantu untuk cek rute dan pos...,positif,aplikasi cukup bantu cek rute posisi bus trans...,"[aplikasi, cukup, bantu, cek, rute, posisi, bu...","[aplikasi, bantu, cek, rute, posisi, bus, tran..."
1,biasa naik jak dan selalu muncul datanya di ap...,negatif,biasa naik jak selalu muncul data aplikasi sek...,"[biasa, naik, jak, selalu, muncul, data, aplik...","[jak, muncul, data, aplikasi, udh, gapernah, m..."
2,jujur ini ngebantu banget sih buat pejuang tra...,positif,jujur ngebantu banget sih buat juang transport...,"[jujur, ngebantu, banget, sih, buat, juang, tr...","[jujur, ngebantu, banget, sih, juang, transpor..."
3,aplikasi transjakarta ini jujur ngebantu bange...,positif,aplikasi transjakarta jujur ngebantu banget si...,"[aplikasi, transjakarta, jujur, ngebantu, bang...","[aplikasi, transjakarta, jujur, ngebantu, bang..."
4,transjakarta adalah alat yang sangat berguna d...,positif,transjakarta alat sangat guna seringkali perlu...,"[transjakarta, alat, sangat, guna, seringkali,...","[transjakarta, alat, seringkali, maksimal, ala..."


In [ ]:
!pip install Sastrawi

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.stem import PorterStemmer
from nltk.stem.snowball import SnowballStemmer

In [ ]:
# 1. Definisikan fungsi stemming untuk teks
def stem_text(words):
    # Membuat objek stemmer dari Sastrawi
    factory = StemmerFactory()
    stemmer = factory.create_stemmer()

    # Pastikan input berupa string atau list kata
    if isinstance(words, list):
        return ' '.join([stemmer.stem(word) for word in words])  # Jika input berupa list
    else:
        return ' '.join([stemmer.stem(word) for word in words.split()])  # Jika input berupa string

# 2. Terapkan fungsi stemming pada kolom 'stopword_removal'
# Nama kolom output disesuaikan menjadi 'stemming' sesuai permintaan Anda
df['stemming'] = df['stopword_removal'].apply(stem_text)

# 3. SOLUSI: Menyaring data agar hanya memunculkan 6 kolom yang Anda inginkan secara berurutan
df_hasil = df[['content_clean', 'sentimen_auto', 'content_processed', 'tokenize', 'stopword_removal', 'stemming']]

# 4. Tampilkan 10 data pertama
df_hasil.head(10)

,content_clean,sentimen_auto,content_processed,tokenize,stopword_removal,stemming
0,aplikasi cukup membantu untuk cek rute dan pos...,positif,aplikasi cukup bantu cek rute posisi bus trans...,"[aplikasi, cukup, bantu, cek, rute, posisi, bu...","[aplikasi, bantu, cek, rute, posisi, bus, tran...",aplikasi bantu cek rute posisi bus transjakart...
1,biasa naik jak dan selalu muncul datanya di ap...,negatif,biasa naik jak selalu muncul data aplikasi sek...,"[biasa, naik, jak, selalu, muncul, data, aplik...","[jak, muncul, data, aplikasi, udh, gapernah, m...",jak muncul data aplikasi udh gapernah muncul s...
2,jujur ini ngebantu banget sih buat pejuang tra...,positif,jujur ngebantu banget sih buat juang transport...,"[jujur, ngebantu, banget, sih, buat, juang, tr...","[jujur, ngebantu, banget, sih, juang, transpor...",jujur ngebantu banget sih juang transportasi f...
3,aplikasi transjakarta ini jujur ngebantu bange...,positif,aplikasi transjakarta jujur ngebantu banget si...,"[aplikasi, transjakarta, jujur, ngebantu, bang...","[aplikasi, transjakarta, jujur, ngebantu, bang...",aplikasi transjakarta jujur ngebantu banget si...
4,transjakarta adalah alat yang sangat berguna d...,positif,transjakarta alat sangat guna seringkali perlu...,"[transjakarta, alat, sangat, guna, seringkali,...","[transjakarta, alat, seringkali, maksimal, ala...",transjakarta alat seringkali maksimal alam bus...
5,biasa naik jak dan jak dan selalu muncul datan...,negatif,biasa naik jak jak selalu muncul data aplikasi...,"[biasa, naik, jak, jak, selalu, muncul, data, ...","[jak, jak, muncul, data, aplikasi, udh, gapern...",jak jak muncul data aplikasi udh gapernah munc...
6,bintang buat kemudahannya skema rutenya jelas ...,positif,bintang buat mudah skema rute jelas banget nge...,"[bintang, buat, mudah, skema, rute, jelas, ban...","[bintang, mudah, skema, rute, banget, ngebantu...",bintang mudah skema rute banget ngebantu bange...
7,membantu bgt utk tracking posisi bis jadi bisa...,positif,bantu bgt utk tracking posisi bis jadi tau est...,"[bantu, bgt, utk, tracking, posisi, bis, jadi,...","[bantu, bgt, utk, tracking, posisi, bis, tau, ...",bantu bgt utk tracking posisi bis tau estimasi...
8,aplikasinya sangat membantu untuk cek rute dan...,positif,aplikasi sangat bantu cek rute jadwal transjak...,"[aplikasi, sangat, bantu, cek, rute, jadwal, t...","[aplikasi, bantu, cek, rute, jadwal, transjaka...",aplikasi bantu cek rute jadwal transjakarta in...
9,ini aplikasi bener ngebantu aku sebagai pendat...,positif,aplikasi bener ngebantu aku datang sih tampil ...,"[aplikasi, bener, ngebantu, aku, datang, sih, ...","[aplikasi, bener, ngebantu, sih, tampil, oke, ...",aplikasi bener ngebantu sih tampil oke mudah p...


In [ ]:
df.to_csv('data_hasil.csv', index=False)
print('File berhasil diunduh')

File berhasil diunduh
